# Test the code from file `bloch.py`

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np
import sympy as sp

# Python auxiliary functions
from IPython.display import display

# Local importations
from moments.bloch import (generate_pauli_basis, generate_gell_mann_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bloch_vector, compute_dm_from_bloch, compute_bloch_norms_from_dm, compute_bloch_norms_from_vector,)

# Basis generation

First, test the functions `generate_pauli_basis`, `generate_gell_mann_basis`, `compute_tensor_basis` and `compute_subset_index_map`.

- `generate_pauli_basis`: generates a numpy array of shape $(4, 2, 2)$ that contains the identity matrix in $\mathbb C^2$ with index $0$ and the Pauli matrices with indices $1, 2, 3$.
- `generate_gell_mann_basis`: Takes as input the dimension $d$ of the system. Generates a numpy array of shape $(d^2, d, d)$ that contains the identity matrix in $\mathbb C^d$ with index $0$ and the generalized Gell-Mann matrices with indices $1, \ldots, d^2$.
- `compute_tensor_basis`: Takes as input a list of numpy arrays that are asumed the local operator bases of a given system. It returns the composite operator basis of the hole system order lexicographically.
- `subset_index_map`: Takes as input a list with the dimension of each local operator hilbert space. Returns a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the indices $i$ of the bloch vector $r_i$ that describe the subsystem $\mathbf M$.

## Pauli basis

We begin by checking that `generate_pauli_basis` generates the Pauli matrices as expected.

Then, for a small system that can be checked by hand we test `compute_tensor_basis` and `compute_subset_index_map`.

In [ ]:
# Generate Pauli basis.
basis = generate_pauli_basis()

# Display generated matrices.
print("Pauli basis shape:", basis.shape)
print("\n Basis elements:")
for s in basis:
    display(sp.Matrix(s))

In [ ]:
# Define a bigger system.
N = 2

local_bases = [basis] * N
local_basis_sizes = [len(lb) for lb in local_bases]

# Compute tensor basis and subset index map.
tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Print the desired tensor basis element. Especify the index to be printed.
idx = 1
print("Tensor basis shape:", tensor_basis.shape)
print("\n Basis element:", idx)
display(sp.Matrix(tensor_basis[idx-1]))

# Print the index representing each subset.
for subset, indices in subset_index_map.items():
    print(subset, "-->", indices)

## Generalized Gell-Mann matrix

We now check that `generate_gell_mann_basis` generates the Gell-Mann matrices for known cases.

We also test that the resulting matrices are a valid generalized Pauli basis. That is, they are hermitian and satisfy $\operatorname{Tr} (\mu_i \, \mu_j) = d \, \delta_{ij}$.

In [ ]:
# Generate Gell-Mann basis.
basis = generate_gell_mann_basis(d = 3)

# Display generated matrices.
print("Pauli basis shape:", basis.shape)
print("\n Basis elements:")
for s in basis:
    display(sp.Matrix(s))

In [ ]:
# Check traces, hermiticity and compute inner products.
traces, products, hermitian = [], [], []
for a in basis:
    traces.append(complex(np.trace(a)))
    hermitian.append(np.allclose(a, a.conj().T))
    for b in basis:
        products.append(complex(np.trace(a @ b.conj().T)))

print(traces)
print(hermitian)
print(products)

# Bloch vector and density-matrix reconstruction

Now we test the utilities `compute_bloch_vector` and `compute_dm_from_bloch`.

- `compute_bloch_vector`: Takes as input the tensor basis and subset index maps of the system, as well as the density matrix. Returns a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the components of the Bloch vector that describe the subsystem $\mathbf M$.
- `compute_dm_from_bloch`: Takes as input the tensor basis and subset index maps of the system, as well as the bloch vector in dictionary format. It returns a numpy array representing the density matrix of the system.

We test the functions in two steps.
1. Generate random Bloch vectors -> compute their density matrices using `compute_dm_from_bloch` -> compute the Bloch vectors using `compute_bloch_vector` -> compute the error.
2. Generate random density matrices -> compute the Bloch vectors using `compute_bloch_vector`. -> compute their density matrices using `compute_dm_from_bloch` -> compute the error.

In [ ]:
# Define system paremeters.
dn = 3
N = 3
d = dn**N

# Compute the appropiate tensor basis and subset index map.
basis = generate_gell_mann_basis(dn)
local_bases = [basis] * N
local_basis_sizes = [len(lb) for lb in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

In [ ]:
# Initialize random generator.
rng = np.random.default_rng()

# Generate random Bloch vectors.
r = {}
for subset, indices in subset_index_map.items():
    r[subset] = rng.normal(size=len(indices))

# Compute density matrices.
rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)

# Compute reconstructed bloch vectors.
r_rec = compute_bloch_vector(tensor_basis, subset_index_map, rho)

# Check differences between initial and reconstructed Bloch vector.
coincide = []
difference = []

for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(r[subset] - r_rec[subset])))
    coincide.append(np.allclose(r[subset], r_rec[subset]))

print("Bloch vectors coincide:", all(coincide))
print("Total difference:", sum(difference))

In [ ]:
# Initialize random generator.
rng = np.random.default_rng()

# Generate random density matrices.
X = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
rho = X @ X.conj().T
tr_val = np.trace(rho).real
rho = rho/tr_val

# Compute Bloch vectors
r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

# Compute reconstructed density matrices.
rho_rec = compute_dm_from_bloch(tensor_basis, subset_index_map, r)

# Check differences between initial and reconstructed density matrices.
print("Density matrices coincide:", np.allclose(rho, rho_rec))
print("Total difference:", np.linalg.norm(rho - rho_rec))

# Bloch norms computation

Now we test the utilities `compute_bloch_norms_from_dm` and `compute_bloch_norms_from_vector`.

- `compute_bloch_norms_from_dm`: Takes as input the tensor basis and subset index maps of the system, as well as the density matrix. Returns a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the norm of the Bloch vector that describe the subsystem $\mathbf M$.
- `compute_bloch_norms_from_vector`: Takes as input the tensor basis and subset index maps of the system, as well as the Bloch vector in dictionary format. Returns a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the norm of the Bloch vector that describe the subsystem $\mathbf M$.

We test the functions in two steps.
1. Generate random Bloch vectors and compute their norms -> compute their density matrices using `compute_dm_from_bloch` -> compute the Bloch norms using `compute_bloch_norms_from_dm` -> compute the error.
2. Generate random Bloch vectors and compute their norms -> compute their density matrices using `compute_dm_from_bloch` -> compute the Bloch vectors using `compute_bloch_vector` -> compute the Bloch norms using `compute_bloch_norms_from_vector` -> compute the error.

In [ ]:
rng = np.random.default_rng()

r = {}
R = {}
for subset, indices in subset_index_map.items():
    r_M = rng.normal(size=len(indices))
    r[subset] = r_M.copy()
    R[subset] = np.linalg.norm(r_M)

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)
r_rec = compute_bloch_vector(tensor_basis, subset_index_map, rho)

R_rho = compute_bloch_norms_from_dm(tensor_basis, subset_index_map, rho)

coincide = []
difference = []

for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(R[subset] - R_rho[subset])))
    coincide.append(np.allclose(R[subset], R_rho[subset]))

print("Bloch norms coincide:", all(coincide))
print("Total difference:", sum(difference))

In [ ]:
R_r = compute_bloch_norms_from_vector(r_rec)

coincide = []
difference = []

for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(R[subset] - R_r[subset])))
    coincide.append(np.allclose(R[subset], R_r[subset]))

print("Bloch norms coincide:", all(coincide))
print("Total difference:", sum(difference))